# MiniMind: learn training from random initialization to inference

This notebook calls the repository's native PyTorch trainer scripts through `colab/minimind_colab.py`. Run each cell in order. First complete the small `micro` path; only then enable the full mini-data `zero` path.

In [ ]:
# Set these to the branch that contains this colab/ directory.
REPOSITORY_URL = 'https://github.com/wangzheng422/minimind.git'
REPOSITORY_REF = 'wzh-main'
ROOT = '/content/minimind'
COLAB_PYTHON = f'{ROOT}/.colab-venv/bin/python'
COLAB_RUNNER = f'{ROOT}/colab/minimind_colab.py'
RUN_ZERO_PROFILE = False
RUN_TOKENIZER_EXPERIMENT = False
USE_GOOGLE_DRIVE = True
DRIVE_DIR = '/content/drive/MyDrive/colab/minimind/'


In [ ]:
!set -e; if test -d "{ROOT}/.git"; then git -C "{ROOT}" fetch --depth 1 origin "{REPOSITORY_REF}" && git -C "{ROOT}" checkout --detach FETCH_HEAD; elif test -e "{ROOT}"; then printf '%s\n' "ERROR: {ROOT} exists but is not a Git checkout" >&2; exit 1; else git clone --depth 1 --branch "{REPOSITORY_REF}" "{REPOSITORY_URL}" "{ROOT}"; fi; cd "{ROOT}"; python "{COLAB_RUNNER}" --root "{ROOT}" setup


In [ ]:
!{COLAB_PYTHON} {COLAB_RUNNER} --root {ROOT} preflight --require-a100


## 1. Tokenizer, model tensors, and next-token labels

The next cells expose the same tokenizer, causal logits, and shifted labels that the trainer uses. `loss: true` means the following token is a supervised target.

In [ ]:
!{COLAB_PYTHON} {COLAB_RUNNER} --root {ROOT} lesson tokenizer --text '语言模型通过预测下一个 token 来学习文本。'
!{COLAB_PYTHON} {COLAB_RUNNER} --root {ROOT} lesson model --profile micro


In [ ]:
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    !{COLAB_PYTHON} {COLAB_RUNNER} --root {ROOT} restore --drive-dir {DRIVE_DIR} --allow-missing
    import subprocess, threading, time
    from pathlib import Path
    Path(ROOT, 'logs').mkdir(exist_ok=True)
    def backup_loop():
        while True:
            with open(f'{ROOT}/logs/drive-backup.log', 'a') as log:
                subprocess.run([COLAB_PYTHON, COLAB_RUNNER, '--root', ROOT, 'backup', '--drive-dir', DRIVE_DIR], cwd=ROOT, stdout=log, stderr=subprocess.STDOUT, check=False)
            time.sleep(600)
    threading.Thread(target=backup_loop, daemon=True).start()
!{COLAB_PYTHON} {COLAB_RUNNER} --root {ROOT} download --all
!{COLAB_PYTHON} {COLAB_RUNNER} --root {ROOT} make-micro --rows 2048
if RUN_TOKENIZER_EXPERIMENT:
    !{COLAB_PYTHON} {COLAB_RUNNER} --root {ROOT} tokenizer-experiment
!{COLAB_PYTHON} {COLAB_RUNNER} --root {ROOT} lesson pretrain-labels --profile micro --rows 24
!{COLAB_PYTHON} {COLAB_RUNNER} --root {ROOT} lesson sft-labels --profile micro --rows 48


## 2. Execute one real optimizer update

This is not a mock: it runs forward, cross-entropy, backward, gradient clipping, AdamW, and a second loss measurement on the same batch.

In [ ]:
!{COLAB_PYTHON} {COLAB_RUNNER} --root {ROOT} one-step --profile micro --stage pretrain
!{COLAB_PYTHON} {COLAB_RUNNER} --root {ROOT} train --profile micro --stage pretrain --resume
!{COLAB_PYTHON} {COLAB_RUNNER} --root {ROOT} infer --profile micro --stage pretrain --prompt '人工智能是' --max-new-tokens 80
if USE_GOOGLE_DRIVE:
    !{COLAB_PYTHON} {COLAB_RUNNER} --root {ROOT} backup --drive-dir {DRIVE_DIR}


In [ ]:
!{COLAB_PYTHON} {COLAB_RUNNER} --root {ROOT} one-step --profile micro --stage sft
!{COLAB_PYTHON} {COLAB_RUNNER} --root {ROOT} train --profile micro --stage sft --resume
!{COLAB_PYTHON} {COLAB_RUNNER} --root {ROOT} infer --profile micro --stage sft --prompt '请解释什么是自注意力机制。' --max-new-tokens 128
if USE_GOOGLE_DRIVE:
    !{COLAB_PYTHON} {COLAB_RUNNER} --root {ROOT} backup --drive-dir {DRIVE_DIR}


## 3. Optional: persist outputs in Google Drive

When enabled before data preparation, Drive is restored first, backed up every 10 minutes, and copied again after each stage. Training always remains on Colab's local disk.

In [ ]:
if USE_GOOGLE_DRIVE:
    !{COLAB_PYTHON} {COLAB_RUNNER} --root {ROOT} backup --drive-dir {DRIVE_DIR}


## 4. Optional: MiniMind Zero reproduction

Set `RUN_ZERO_PROFILE = True` only after the micro route succeeds. This uses the complete official mini JSONL files and the repository's 768-hidden, 8-layer configuration.

In [ ]:
if RUN_ZERO_PROFILE:
    !{COLAB_PYTHON} {COLAB_RUNNER} --root {ROOT} lesson model --profile zero
    !{COLAB_PYTHON} {COLAB_RUNNER} --root {ROOT} train --profile zero --stage pretrain --resume
    !{COLAB_PYTHON} {COLAB_RUNNER} --root {ROOT} infer --profile zero --stage pretrain --prompt '机器学习是一种' --max-new-tokens 128
    if USE_GOOGLE_DRIVE:
        !{COLAB_PYTHON} {COLAB_RUNNER} --root {ROOT} backup --drive-dir {DRIVE_DIR}
    !{COLAB_PYTHON} {COLAB_RUNNER} --root {ROOT} train --profile zero --stage sft --resume
    !{COLAB_PYTHON} {COLAB_RUNNER} --root {ROOT} infer --profile zero --stage sft --prompt '请用通俗语言解释 Transformer。' --max-new-tokens 256
    if USE_GOOGLE_DRIVE:
        !{COLAB_PYTHON} {COLAB_RUNNER} --root {ROOT} backup --drive-dir {DRIVE_DIR}
